#Модели предсказания содержания элемента в стали

Цифровой советчик должен выдавать сталевару рекомендации по количеству и наименованию ферросплавов для того, чтобы _элемент_Final лежало в интервале (Элемент_LowerLimit, Элемент_Target)

Для начала определим признаки и целевые переменные

In [ ]:
elements = ['Cr', 'Mo', 'Ni', 'V', 'W']
target_cols = [f'{el}_Final' for el in elements]

exclude_cols = ['HeatNo', 'Date'] + target_cols

feature_cols = [c for c in data.columns if c not in exclude_cols]

print(f"Всего признаков X: {len(feature_cols)}")
print(feature_cols)

alloy_cols    = [c for c in feature_cols if c.startswith('LFVD_')]
base_features = [c for c in feature_cols if c not in alloy_cols]

print(f"\nБазовые признаки ({len(base_features)} шт.): {base_features}")
print(f"Добавки ({len(alloy_cols)} шт.): {alloy_cols}")

Всего признаков X: 74
['TotalIngotsWeight', 'PouringScrap', 'OtherScrap', 'Last_EOP', 'Cr_Last_EOP', 'Ni_Last_EOP', 'Mo_Last_EOP', 'V_Last_EOP', 'W_Last_EOP', 'Cr_LowerLimit', 'Cr_Target', 'Cr_UpperLimit', 'Ni_LowerLimit', 'Ni_Target', 'Ni_UpperLimit', 'Mo_LowerLimit', 'Mo_Target', 'Mo_UpperLimit', 'V_LowerLimit', 'V_Target', 'V_UpperLimit', 'W_LowerLimit', 'W_Target', 'W_UpperLimit', 'LFVD_AlBloki', 'LFVD_AlGran', 'LFVD_Boksit', 'LFVD_CaO', 'LFVD_CASIfi13', 'LFVD_Cfi13', 'LFVD_EPŽŽlindra', 'LFVD_FeAl', 'LFVD_FeCrA', 'LFVD_FeCrC', 'LFVD_FeCrCSi', 'LFVD_FeCrC51', 'LFVD_FeMnC', 'LFVD_FeMo', 'LFVD_FeS', 'LFVD_FeSi', 'LFVD_FeV', 'LFVD_FeW72', 'LFVD_KarboritMleti', 'LFVD_MnMet', 'LFVD_NiGran', 'LFVD_Polymox', 'LFVD_SŽica', 'LFVD_SiMn', 'LFVD_SLAGMAG65B', 'LFVD_AlŽica', 'LFVD_AlOplašèenaŽica', 'LFVD_BelaŽlindra', 'LFVD_KalcijevKarbid', 'LFVD_SintŽlindra', 'PV_PT181', 'PV_OH252', 'PV_OH255', 'PV_EMCR', 'PV_VCMO230', 'PV_31CRV3', 'PV_OCR12', 'PV_OCR12SP', 'PV_CRV', 'PV_OCR12VM', 'PV_UTOPMO2', 

Затем сформируем обучающую и тестовые выборки

In [ ]:
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

needed = feature_cols + target_cols

df_model = data[needed].dropna().copy()
print(f"Размер датасета для моделирования: {len(df_model)} плавок")
print(f"(Отброшено из-за пропусков: {len(data) - len(df_model)} плавок)")

X = df_model[feature_cols]
X_train, X_test, idx_train, idx_test = train_test_split(X, df_model.index, test_size=0.2, random_state=42)
print(f"Обучающая выборка: {len(X_train)} плавок")
print(f"Тестовая выборка:  {len(X_test)} плавок")

Размер датасета для моделирования: 3212 плавок
(Отброшено из-за пропусков: 13 плавок)
Обучающая выборка: 2569 плавок
Тестовая выборка:  643 плавок


Обучение бустинга

In [ ]:
models = {}
for el in elements:
    y_train = df_model.loc[idx_train, f'{el}_Final']
    y_test  = df_model.loc[idx_test,  f'{el}_Final']
    model = LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        random_state=42,
        verbose=-1
    )
    model.fit(X_train, y_train)
    models[el] = model
    y_pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    me = np.mean(y_pred - y_test.values)
    print(f"Модель для {el}_Final")
    print(f"  MAE  (ср. абс. ошибка)     = {mae:.5f} %")
    print(f"  R²   (коэф. детерминации)  = {r2:.4f}")
    print(f"  ME   (среднее смещение)    = {me:+.5f} % \n")

Модель для Cr_Final
  MAE  (ср. абс. ошибка)     = 0.07214 %
  R²   (коэф. детерминации)  = 0.7879
  ME   (среднее смещение)    = -0.00462 % 

Модель для Mo_Final
  MAE  (ср. абс. ошибка)     = 0.01438 %
  R²   (коэф. детерминации)  = 0.8256
  ME   (среднее смещение)    = -0.00001 % 

Модель для Ni_Final
  MAE  (ср. абс. ошибка)     = 0.00932 %
  R²   (коэф. детерминации)  = 0.9080
  ME   (среднее смещение)    = -0.00012 % 

Модель для V_Final
  MAE  (ср. абс. ошибка)     = 0.01630 %
  R²   (коэф. детерминации)  = 0.9416
  ME   (среднее смещение)    = -0.00211 % 

Модель для W_Final
  MAE  (ср. абс. ошибка)     = 0.01042 %
  R²   (коэф. детерминации)  = 0.8670
  ME   (среднее смещение)    = +0.00033 % 



У всех пяти моделей ME отрицательный (кроме W). Это означает что модели в среднем чуть занижают предсказание. Для советчика это значит: модель будет думать что хрома получится чуть меньше чем на самом деле, и порекомендует чуть больше феррохрома чем нужно. Это безопасная сторона ошибки — лучше немного перерасход ферросплава чем брак по нижней границе. Значения ME очень маленькие (0.004% для Cr, для остальных ещё меньше) — систематического смещения практически нет, что хорошо.

У всех моделей ME на порядок меньше MAE — это главное. Означает что нет систематического перекоса: модель не всегда ошибается в одну сторону, ошибки случайные. Отрицательный знак ME (почти у всех) — безопасная сторона для советчика, как обсудили выше.

Посчитаем ширину коридоров (допустимых значений)

In [ ]:
print(f"{'Элемент':<6} {'UpperLimit-LowerLimit':>22} {'Target-LowerLimit':>18}")
for el in elements:
    full    = (data[f'{el}_UpperLimit'] - data[f'{el}_LowerLimit']).mean()
    adviser = (data[f'{el}_Target']     - data[f'{el}_LowerLimit']).mean()
    print(f"  {el:<6} {full:>22.4f}% {adviser:>18.4f}%")

Элемент  UpperLimit-LowerLimit  Target-LowerLimit
  Cr                     0.6157%             0.2640%
  Mo                     0.1366%             0.0309%
  Ni                     0.2814%             0.0000%
  V                      0.1176%             0.0411%
  W                      0.1803%             0.0000%


Сохранение модели

In [ ]:
import joblib

joblib.dump(models, 'models.pkl')
print("Модели сохранены в models.pkl")

Модели сохранены в models.pkl


Импорт моделей

In [ ]:
# import joblib
# models = joblib.load('models.pkl')
# print("Модели загружены")

#Советчик

In [ ]:
TARGET_TOLERANCE = 0.05

def adviser(known_data: dict, lower_limits: dict, targets: dict, upper_limits: dict,
            models: dict, alloy_combinations: pd.DataFrame,
            base_features: list, alloy_cols: list):
    """
    Цифровой советчик сталевара
    Перебирает все исторические комбинации ферросплавов и ищет ту, при которой модель предсказывает попадание нормируемых элементов в
    допуск. Логика проверки допуска по каждому элементу:
      - Стандартная проверка: LowerLimit <= pred <= Target + TARGET_TOLERANCE
      - Если LowerLimit == Target — найти комбинацию ферросплавов, удовлетворяющую таким условиям, практически невозможно. А потому будет достаточно
        проверить соответствие по браку (а не по экономической эффективности). Проверяем только чтобы LowerLimit <= pred <= UpperLimit

    Параметры:
    known_data         : dict — известные параметры плавки ДО добавки ферросплавов, их вводит сталевар . Это характеристики
                         полупродукта и добавляемого лома, а также вес плавки
    lower_limits       : dict {элемент: нижняя граница допуска}
    targets            : dict {элемент: целевое значение}
    upper_limits       : dict {элемент: верхняя граница допуска}
    models             : dict {элемент: обученная модель} - 5 обученных моделей
    alloy_combinations : DataFrame — комбинации добавок (3200 рецептов из нашего датасета)
    base_features      : list — названия базовых признаков (состав полупродукта, вес, лом)
    alloy_cols         : list — названия ферросплавов

    Возвращает dict:
    status          : 'OK' или 'NO_SOLUTION'
    alloys          : dict — ненулевые дозы ферросплавов (кг)
    predicted_final : dict — прогноз состава стали по каждому элементу
    total_mass_kg   : суммарная масса рекомендованных добавок
    source_heat_no  : HeatNo плавки откуда взят рецепт
    """

    for el in elements:
        known_data[f'{el}_LowerLimit'] = lower_limits[el]
        known_data[f'{el}_Target']     = targets[el]
        known_data[f'{el}_UpperLimit'] = upper_limits[el]
    base_row = pd.Series(known_data)

    best = {'mass': float('inf'), 'alloys':  None, 'preds': None, 'heat_no': None}

    for index, combo in alloy_combinations.iterrows():
        full_row = pd.concat([base_row, combo[alloy_cols]])
        x  = pd.DataFrame([full_row.values], columns=feature_cols)
        predicted = {}
        for el, model in models.items():
            predicted[el] = model.predict(x)[0]

        in_range = True
        for el in models:
            if lower_limits[el] == targets[el]:
                if predicted[el] > upper_limits[el]:
                    in_range = False
                    break
                continue
            right_bound = min(targets[el] + TARGET_TOLERANCE, upper_limits[el])
            if not (lower_limits[el] <= predicted[el] <= right_bound):
                in_range = False
                break

        if in_range:
            total_mass = combo[alloy_cols].sum()
            if total_mass < best['mass']:
                best['mass']    = total_mass
                best['alloys']  = combo[alloy_cols].copy()
                best['preds']   = predicted.copy()
                best['heat_no'] = data.loc[index, 'HeatNo']

    if best['alloys'] is None:
        return {"status":  "NO_SOLUTION",
                "message": "Не найдено ни одной допустимой комбинации"}

    recommended = best['alloys'][best['alloys'] > 0.001]

    return {
        "status": "OK",
        "alloys": recommended.to_dict(),
        "predicted_final": {el: round(v, 4) for el, v in best['preds'].items()},
        "total_mass_kg": round(recommended.sum(), 1),
        "source_heat_no": best['heat_no']}

Комбинаций для перебора: 3222


In [ ]:
TARGET_TOLERANCE = 0.025

def adviser(known_data: dict,
            models: dict, alloy_combinations: pd.DataFrame,
            base_features: list, alloy_cols: list):
    """
    Цифровой советчик сталевара
    Перебирает все исторические комбинации ферросплавов и ищет ту, при которой модель предсказывает попадание нормируемых элементов в
    допуск. Логика проверки допуска по каждому элементу:
      - Стандартная проверка: LowerLimit <= pred <= Target + TARGET_TOLERANCE
      - Если LowerLimit == Target — найти комбинацию ферросплавов, удовлетворяющую таким условиям, практически невозможно. А потому будет достаточно
        проверить соответствие по браку (а не по экономической эффективности). Проверяем только чтобы LowerLimit <= pred <= UpperLimit

    Параметры:
    known_data         : dict — все параметры которые вводит сталевар:
                         - характеристики полупродукта и лома (состав, вес)
                         - лимиты марки стали (LowerLimit, Target, UpperLimit для каждого элемента)
    models             : dict {элемент: обученная модель} - 5 обученных моделей
    alloy_combinations : DataFrame — комбинации добавок (3200 рецептов из нашего датасета)
    base_features      : list — названия базовых признаков (состав полупродукта, вес, лом)
    alloy_cols         : list — названия ферросплавов

    Возвращает dict:
    status          : 'OK' или 'NO_SOLUTION'
    alloys          : dict — ненулевые дозы ферросплавов (кг)
    predicted_final : dict — прогноз состава стали по каждому элементу
    total_mass_kg   : суммарная масса рекомендованных добавок
    source_heat_no  : HeatNo плавки откуда взят рецепт
    """

    base_row = pd.Series(known_data)

    best = {'mass': float('inf'), 'alloys': None, 'preds': None, 'heat_no': None}

    for index, combo in alloy_combinations.iterrows():
        full_row = pd.concat([base_row, combo[alloy_cols]])
        x = pd.DataFrame([full_row.values], columns=feature_cols)

        predicted = {}
        for el, model in models.items():
            predicted[el] = model.predict(x)[0]

        in_range = True
        for el in models:
            if known_data[f'{el}_LowerLimit'] == known_data[f'{el}_Target']:
                if not (known_data[f'{el}_LowerLimit'] <= predicted[el] <= known_data[f'{el}_UpperLimit']):
                    in_range = False
                    break
                continue

            right_bound = min(known_data[f'{el}_Target'] + TARGET_TOLERANCE, known_data[f'{el}_UpperLimit'])
            if not (known_data[f'{el}_LowerLimit'] <= predicted[el] <= right_bound):
                in_range = False
                break

        if in_range:
            total_mass = combo[alloy_cols].sum()
            if total_mass == 0:
                continue
            if total_mass < best['mass']:
                best['mass']    = total_mass
                best['alloys']  = combo[alloy_cols].copy()
                best['preds']   = predicted.copy()
                best['heat_no'] = data.loc[index, 'HeatNo']

    if best['alloys'] is None:
        return {"status":  "NO_SOLUTION",
                "message": "Не найдено ни одной допустимой комбинации"}

    recommended = best['alloys'][best['alloys'] > 0.001]

    return {
        "status":          "OK",
        "alloys":          recommended.to_dict(),
        "predicted_final": {el: round(v, 4) for el, v in best['preds'].items()},
        "total_mass_kg":   round(recommended.sum(), 1),
        "source_heat_no":  best['heat_no']
    }

#Тестирование

In [ ]:
alloy_combinations_full = (data[alloy_cols].drop_duplicates().reset_index(drop=True))

good_examples = df_model.loc[idx_test][(
    df_model.loc[idx_test]['Cr_Last_EOP'] < df_model.loc[idx_test]['Cr_Target']) &
    (df_model.loc[idx_test]['Mo_Last_EOP'] < df_model.loc[idx_test]['Mo_Target']) &
    (df_model.loc[idx_test]['V_Last_EOP']  < df_model.loc[idx_test]['V_Target'])].head(3)

for i, (row_idx, row) in enumerate(good_examples.iterrows(), 1):
    known_data_example = {f: row[f] for f in base_features}

    result = adviser(
        known_data         = known_data_example,
        models             = models,
        alloy_combinations = alloy_combinations_full,
        base_features      = base_features,
        alloy_cols         = alloy_cols
    )

    heat_id = data.loc[row_idx, 'HeatNo'] if 'HeatNo' in data.columns else row_idx
    print(f"\nПЛАВКА #{i}  (HeatNo: {heat_id})")

    print("\nСостав полупродукта (Last_EOP):")
    for el in elements:
        print(f"  {el}: {row[el + '_Last_EOP']:.4f}%")
    print(f"  Вес плавки: {row['TotalIngotsWeight']:.0f} кг")

    print("\nТребования к марке стали:")
    for el in elements:
        ll = known_data_example[f'{el}_LowerLimit']
        tg = known_data_example[f'{el}_Target']
        ul = known_data_example[f'{el}_UpperLimit']
        if ll == tg:
            print(f"  {el}: проверка по браку (до UpperLimit={ul:.4f}%)")
        else:
            print(f"  {el}: {ll:.4f}% → {tg:.4f}%  (до {tg + TARGET_TOLERANCE:.4f}% с допуском)")

    if result['status'] == 'NO_SOLUTION':
        print(result['message'])
    else:
        print(f"\nРЕКОМЕНДАЦИЯ (рецепт из плавки HeatNo: {result['source_heat_no']}):")
        print(f"  {'Ферросплав':<25} {'Масса (кг)':>10}")
        for alloy, kg in result['alloys'].items():
            print(f"  {alloy:<25} {kg:>10.1f}")
        print(f"  {'ИТОГО':<25} {result['total_mass_kg']:>10.1f}")

        print("\nПРОГНОЗ состава стали после добавки:")
        print(f"  {'Эл.':<5} {'LowerLimit':>12} {'Прогноз':>12} {'Target':>12} {'UpperLimit':>12}  Статус")
        for el in elements:
            ll   = known_data_example[f'{el}_LowerLimit']
            tg   = known_data_example[f'{el}_Target']
            ul   = known_data_example[f'{el}_UpperLimit']
            pred = result['predicted_final'][el]
            if ll == tg:
                ok = "OK" if ll <= pred <= ul else f"вне коридора [{ll:.4f}, {ul:.4f}]"
                print(f"  {el:<5} {ll:>12.4f} {pred:>12.4f} {tg:>12.4f} {ul:>12.4f}  {ok}")
            elif ll <= pred <= tg + TARGET_TOLERANCE:
                ok = "OK" if pred <= tg else f"OK (с допуском +{TARGET_TOLERANCE})"
                print(f"  {el:<5} {ll:>12.4f} {pred:>12.4f} {tg:>12.4f} {ul:>12.4f}  {ok}")
            elif pred < ll:
                print(f"  {el:<5} {ll:>12.4f} {pred:>12.4f} {tg:>12.4f} {ul:>12.4f}  ↓ ниже на {ll - pred:.4f}%")
            else:
                print(f"  {el:<5} {ll:>12.4f} {pred:>12.4f} {tg:>12.4f} {ul:>12.4f}  ↑ выше на {pred - tg:.4f}%")
    print("\n")


ПЛАВКА #1  (HeatNo: 88824)

Состав полупродукта (Last_EOP):
  Cr: 11.1100%
  Mo: 0.5500%
  Ni: 0.1200%
  V: 0.2710%
  W: 0.0300%
  Вес плавки: 49440 кг

Требования к марке стали:
  Cr: 11.1000% → 11.4600%  (до 11.4850% с допуском)
  Mo: 0.7000% → 0.7300%  (до 0.7550% с допуском)
  Ni: проверка по браку (до UpperLimit=0.2500%)
  V: 0.7200% → 0.7600%  (до 0.7850% с допуском)
  W: проверка по браку (до UpperLimit=0.1500%)

РЕКОМЕНДАЦИЯ (рецепт из плавки HeatNo: 84037):
  Ферросплав                Масса (кг)
  LFVD_AlBloki                     3.0
  LFVD_Boksit                     80.0
  LFVD_CaO                       400.0
  LFVD_CASIfi13                   13.9
  LFVD_Cfi13                       8.7
  LFVD_EPŽŽlindra                120.0
  LFVD_FeMo                       70.0
  LFVD_FeSi                      100.0
  LFVD_FeV                       320.0
  LFVD_KarboritMleti              50.0
  LFVD_SiMn                       20.0
  LFVD_AlOplašèenaŽica             3.9
  LFVD_KalcijevKarbid